## Fuente de los datos

Dataset diario de demanda, generación y precios eléctricos de España (2014-2018), 
recopilado de ESIOS, la plataforma de REE (Red Eléctrica Española), el operador 
del sistema eléctrico español. El dataset mantiene el formato original de la fuente, 
por lo que las columnas `geoid`/`geoname` vienen vacías en la mayoría de las filas 
(esto es intencional, según la documentación del dataset, para facilitar agregar 
nuevos datos con el mismo formato).

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

df = pd.read_csv('spain_energy_market.csv')
df.head(10)

,datetime,id,name,geoid,geoname,value
0,2014-01-01 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,25.280833
1,2014-01-02 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,39.924167
2,2014-01-03 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,4.992083
3,2014-01-04 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,4.091667
4,2014-01-05 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,13.587500
5,2014-01-06 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,47.885417
6,2014-01-07 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,41.207500
7,2014-01-08 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,49.022083
8,2014-01-09 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,56.202917
9,2014-01-10 23:00:00,600,Precio mercado SPOT Diario ESP,3.0,España,50.277083


In [24]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 40212 entries, 0 to 40211
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   datetime  40212 non-null  str    
 1   id        40212 non-null  int64  
 2   name      34734 non-null  str    
 3   geoid     10956 non-null  float64
 4   geoname   10956 non-null  str    
 5   value     40212 non-null  float64
dtypes: float64(2), int64(1), str(3)
memory usage: 1.8 MB


In [25]:
df['name'].value_counts(dropna=False)

name
NaN                                                                       5478
Precio mercado SPOT Diario ESP                                            1826
Precio mercado SPOT Diario FRA                                            1826
Precio mercado SPOT Diario POR                                            1826
Energía asignada en Mercado SPOT Diario España                            1826
Energía asignada en Mercado SPOT Diario Francia                           1826
Generación programada PBF Gas Natural Cogeneración                        1826
Generación programada PBF UGH + no UGH                                    1826
Generación programada PBF Solar fotovoltaica                              1826
Demanda real                                                              1825
Demanda programada PBF total                                              1825
Generación programada PBF total                                           1825
Generación programada PBF Eólica               

In [26]:
precio_es = df[df['name'] == 'Precio mercado SPOT Diario ESP'][['datetime', 'value']]
precio_es = precio_es.rename(columns={'value': 'precio_eur_mwh'})

demanda_es = df[df['name'] == 'Demanda real'][['datetime', 'value']]
demanda_es = demanda_es.rename(columns={'value': 'demanda_mw'})

print(precio_es.shape)
print(demanda_es.shape)
precio_es.head()

(1826, 2)
(1825, 2)


,datetime,precio_eur_mwh
0,2014-01-01 23:00:00,25.280833
1,2014-01-02 23:00:00,39.924167
2,2014-01-03 23:00:00,4.992083
3,2014-01-04 23:00:00,4.091667
4,2014-01-05 23:00:00,13.587500


In [27]:
precio_es['datetime'] = pd.to_datetime(precio_es['datetime'])
demanda_es['datetime'] = pd.to_datetime(demanda_es['datetime'])

serie_es = pd.merge(precio_es, demanda_es, on='datetime', how='outer')
serie_es = serie_es.sort_values('datetime')

print(serie_es.shape)
print(serie_es.isnull().sum())
serie_es.head()

(1826, 3)
datetime          0
precio_eur_mwh    0
demanda_mw        1
dtype: int64


,datetime,precio_eur_mwh,demanda_mw
0,2014-01-01 23:00:00,25.280833,28191.597222
1,2014-01-02 23:00:00,39.924167,28465.180556
2,2014-01-03 23:00:00,4.992083,26860.493056
3,2014-01-04 23:00:00,4.091667,25333.597222
4,2014-01-05 23:00:00,13.587500,23905.541667


In [28]:
serie_es[serie_es['demanda_mw'].isnull()]

,datetime,precio_eur_mwh,demanda_mw
1825,2018-12-31 23:00:00,63.454583,NaN


Interpolar significa asumir que, si la demanda pasó de 100 a 120 en dos días, probablemente el día 2 haya estado en el medio, algo así como 110 — como si trazaras una línea recta entre el punto del día 1 y el punto del día 3, y el valor faltante fuera exactamente el punto de esa línea que corresponde al día 2.

interpolate(method='time') hace exactamente esto, pero siendo más preciso con las fechas reales (si el hueco no está exactamente "a mitad de camino" en el tiempo entre sus vecinos, ajusta la proporción según cuántas horas/días pasaron realmente).

In [29]:
serie_es = serie_es.set_index('datetime')
serie_es['demanda_mw'] = serie_es['demanda_mw'].ffill()

print(serie_es.isnull().sum())
serie_es.tail()

precio_eur_mwh    0
demanda_mw        0
dtype: int64


,precio_eur_mwh,demanda_mw
datetime,,
2018-12-27 23:00:00,63.785417,28624.194444
2018-12-28 23:00:00,57.048333,26632.840278
2018-12-29 23:00:00,58.493750,25255.055556
2018-12-30 23:00:00,61.795000,25908.576389
2018-12-31 23:00:00,63.454583,25908.576389
